In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

brfss_final= pd.read_csv("/content/brfss_final_harmonized.csv")
uganda_final=pd.read_csv("/content/uganda_final_harmonized.csv")
geda_final=pd.read_csv("/content/geda_final_harmonized.csv")


In [ ]:
#Define features
features =["age_group", "sex_male", "bmi_group", "hypertension", "current_smoker", "physically_active"]
target ="diabetes_binary"

In [ ]:
#Function
def run_logistic_model(df, dataset_name):
    data=df.dropna(subset=[target]).copy()
    x= data[features]
    y=data[target]

    x_train, x_test, y_train, y_test =train_test_split(
    x,y, test_size=0.3, stratify=y, random_state=42
    )

    preprocessor =ColumnTransformer(
         transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), features)

         ]

    )

    model = Pipeline([
       ("prep", preprocessor),
       ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))

    ])

    model.fit(x_train, y_train)

    proba =model.predict_proba(x_test)[:,1]
    pred=(proba >=0.5).astype(int)

    results={
        "Dataset": dataset_name,
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    }

    return results

In [ ]:
brfss_results = run_logistic_model(brfss_final, "BRFSS")
uganda_results =run_logistic_model(uganda_final, "Uganda")
geda_results =run_logistic_model(geda_final, "GEDA")

results_df_h =pd.DataFrame([brfss_results, uganda_results,geda_results])
results_df_h


,Dataset,AUC,Precision,Recall,F1
0,BRFSS,0.785146,0.275475,0.728067,0.399712
1,Uganda,0.625276,0.022581,0.500000,0.043210
2,GEDA,0.793688,0.217936,0.719745,0.334567


After stricter harmonization of age and BMI into grouped variables, internal model performance reamined broadly similar to the earlier exploratory runs. BRFSS and GEDA continued to show moderate to good discrimination while Uganda remained substantially weaker. This uggets that the observed performance differences are not primarily driven by the age/BMI representation but more likely by differences in sample size, class balance and outcome structure across datasets.